# 03 — HLLSet core: IICA and the lattice

Substitute for the discharged core notebooks. The HLLSet is an element of
the IICA lattice (Idempotent, Immutable, Content-Addressed): a bitmap over
`B` = 1024 registers × 32 bits, keyed by content, operated on by union/
intersection/difference, and collapsible to one sketch per collection.


In [2]:
:dep hllset-contracts = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-contracts" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-core" }


In [3]:
use hllset_contracts::{token_in_bytes, BitAddress};
use hllset_core::content_addr::{content_key_from_tokens, original_key_from_tokens};
use hllset_core::HLLSet;
fn same(a: &HLLSet, b: &HLLSet) -> bool {
    a.difference(b).popcount() == 0 && b.difference(a).popcount() == 0
}
println!("hllset-core loaded");


hllset-core loaded


---
## Content is identity

The key is the content: sorted tokens, NUL-joined, SHA-1, with the prefix
as the type — `o:` original, `h:` heterogeneous.


In [4]:
let tokens: Vec<Vec<u8>> = vec![b"alpha".to_vec(), b"beta".to_vec()];
let h_key = content_key_from_tokens(&tokens);
let o_key = original_key_from_tokens(&tokens);
println!("h-key: {}", h_key);
println!("o-key: {}", o_key);
println!("idempotent (same tokens, same key): {}",
    h_key == content_key_from_tokens(&tokens));


h-key: h:9680a753e0c209e7de3b726699ef7dd1996f55cf
o-key: o:9680a753e0c209e7de3b726699ef7dd1996f55cf
idempotent (same tokens, same key): true


---
## Lattice laws

Union, intersection, difference — and the laws that hold exactly.


In [5]:
let a: HLLSet = HLLSet::from_tokens([token_in_bytes(1), token_in_bytes(2)]);
let b: HLLSet = HLLSet::from_tokens([token_in_bytes(2), token_in_bytes(3)]);
println!("|A|={} |B|={} |A∪B|={} |A∩B|={} |A\\B|={}",
    a.popcount(), b.popcount(), a.union(&b).popcount(),
    a.intersection(&b).popcount(), a.difference(&b).popcount());
println!("A ∪ A = A:      {}", same(&a.union(&a), &a));
println!("A ∩ A = A:      {}", same(&a.intersection(&a), &a));
println!("A \\ A = ∅:      {}", a.difference(&a).popcount() == 0);
println!("absorption A∪(A∩B)=A: {}", same(&a.union(&a.intersection(&b)), &a));


|A|=2 |B|=2 |A∪B|=3 |A∩B|=1 |A\B|=1
A ∪ A = A:      true
A ∩ A = A:      true
A \ A = ∅:      true
absorption A∪(A∩B)=A: true


---
## Sub-lattice collapse

Any collection of HLLSets collapses to its join — one sketch, one tensor slice.


In [6]:
let gens: Vec<HLLSet> = (0..4u32)
    .map(|i| HLLSet::from_tokens([token_in_bytes(i), token_in_bytes(i + 1)]))
    .collect();
let join: HLLSet = HLLSet::union_all(gens.iter().cloned());
println!("4 generators collapse to one sketch, popcount {}", join.popcount());


4 generators collapse to one sketch, popcount 5


---
## Serialization and atoms

Round-trip through bytes; the sketch decomposes uniquely into active atoms.


In [7]:
let bytes: Vec<u8> = a.to_bytes();
let a2: HLLSet = HLLSet::from_bytes(&bytes).unwrap();
println!("serialization round-trip: {}", same(&a, &a2));
let atoms: Vec<u32> = a.bit_addresses().iter().map(|x| x.bit()).collect();
println!("atoms == popcount: {} ({} atoms)", atoms.len() as u64 == a.popcount(), atoms.len());
for bit in atoms.iter().take(3) {
    let addr = BitAddress::new(*bit);
    println!("  atom bit {} = (reg {}, tz {})", bit, addr.reg(), addr.tz());
}


serialization round-trip: true
atoms == popcount: true (2 atoms)
  atom bit 576 = (reg 18, tz 0)
  atom bit 29763 = (reg 930, tz 3)


()

---
## Cardinality estimate

popcount is exact bit count; cardinality is the HLL estimate (monotone).


In [8]:
let mut h: HLLSet = HLLSet::new();
for id in 0..200u32 { h.add_bit(BitAddress::of_token(&token_in_bytes(id)).bit()); }
println!("200 distinct tokens: popcount {}, cardinality estimate {:.1}",
    h.popcount(), h.cardinality());


200 distinct tokens: popcount 193, cardinality estimate 199.0
